In [2]:
# import datasets

In [3]:
# dataset = datasets.load_dataset("iwslt2017", "iwslt2017-en-de", trust_remote_code=True)

In [2]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig

In [3]:
# bnb_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForSeq2SeqLM.from_pretrained("./models/nllb_teacher").eval()
tokenizer = AutoTokenizer.from_pretrained("./models/nllb_teacher")

In [5]:
text = [
    "When the old man opened the small wooden door at the end of the garden, he saw a narrow path leading through the wet grass toward a quiet river, where a little girl in a red coat was feeding ducks and laughing because the wind kept turning her yellow umbrella inside out.",
    "For he did not know that beyond the lake he called home lies a deeper, darker ocean green, where waves are both wilder and more serene, to its ports I've been, to its ports I've been."
]
# text = "For he did not know that beyond the lake he called home lies a deeper, darker ocean green, where waves are both wilder and more serene, to its ports I've been, to its ports I've been."
# text = "By the time the engineer realized that the lead pipe she had been asked to wind around the old machine was actually meant to record the pressure changes rather than carry water, the supervisor, who had been present during the confusing meeting, refused to permit anyone to object until the final report was read aloud."
# text = "EX-View is a proprietary SONY technology in which the P/N junction of each photodiode in the CCD matrix is specially fabricated to have much better photon-to-electron conversion efficiency."
inputs = tokenizer(text, return_tensors="pt", padding=True).to(model.device)

In [6]:
tokenizer.batch_decode(inputs["input_ids"], skip_special_tokens=False)

['eng_Latn When the old man opened the small wooden door at the end of the garden, he saw a narrow path leading through the wet grass toward a quiet river, where a little girl in a red coat was feeding ducks and laughing because the wind kept turning her yellow umbrella inside out.</s>',
 "eng_Latn For he did not know that beyond the lake he called home lies a deeper, darker ocean green, where waves are both wilder and more serene, to its ports I've been, to its ports I've been.</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>"]

In [7]:
START_TOKEN = tokenizer.convert_tokens_to_ids("rus_Cyrl")
# START_TOKEN = tokenizer.convert_tokens_to_ids("eng_Latn")

In [17]:
@torch.no_grad()
def translate_with_latency(model, tokenizer, source: str, max_len: int = 100, k: int = 0, speed: int = 1) -> str:
    inputs = tokenizer(source, return_tensors="pt", padding=True).to(model.device)
    if k <= 1:
        return model.generate(**inputs, forced_bos_token_id=START_TOKEN, max_length=max_len)

    latency_inputs = inputs["input_ids"][:, :k]
    latency_attention_mask = inputs["attention_mask"][:, :k]
    target_tokens = torch.tensor([[model.config.decoder_start_token_id, START_TOKEN]], dtype=torch.long, device=model.device)
    prefix_len = latency_inputs.shape[-1]
    i = 1

    while True:
        outputs = torch.argmax(model(
            input_ids=torch.concat([latency_inputs, torch.tensor([[tokenizer.eos_token_id]], device=model.device)], dim=-1),
            attention_mask=torch.concat([latency_attention_mask, torch.tensor([[1]], dtype=torch.long, device=model.device)], dim=-1),
            decoder_input_ids=target_tokens,
            use_cache=False
        ).logits[:, -1], dim=-1).unsqueeze(-1)

        if outputs[:, -1].item() != tokenizer.eos_token_id:
            target_tokens = torch.concat([target_tokens, outputs], dim=-1)

        print(f"Iteration {i}")
        print(f"\tInput: {tokenizer.batch_decode(latency_inputs, skip_special_tokens=True)[0]}")
        print(f"\tTarget: {tokenizer.batch_decode(target_tokens, skip_special_tokens=True)[0]}")
        # print(f"\tGenerated token: {tokenizer.batch_decode(outputs, skip_special_tokens=True)}")
        i += 1

        if k <= prefix_len:
            k += speed
            latency_inputs = inputs["input_ids"][:, :k]
            latency_attention_mask = inputs["attention_mask"][:, :k]

        prefix_len = latency_inputs.shape[-1]

        if outputs[:, -1].item() == tokenizer.eos_token_id and k >= inputs["input_ids"].shape[-1] or prefix_len >= max_len:
            break

    return tokenizer.batch_decode(target_tokens, skip_special_tokens=True)

In [18]:
translation = translate_with_latency(model, tokenizer, text, 100, 0, 1)

In [19]:
translation

tensor([[     2, 256147,  61275,   2704,  51769,  81608, 248103,  74010,  46634,
          79847,   8084, 248088,  22569, 172229, 248293,    191, 227885,  44498,
         248079,   3430,  63007,  13620,  22663,   9318, 180708, 248079,   2381,
           1351,  74672,  25219,    800,  78459, 248478,  13274,   3466,    137,
           1651,  71122,   1512,   1140, 248079,  29039,  74010,  48176, 183513,
          13202,    191,   6473,   3815, 248118, 165322,    205,  10478, 157424,
          26655,    567,    213,  13699,   2935,   1984, 248079,  58221,   3413,
           2381,   1796,  45811, 131398,  19644, 236351,  38458,  58501,  63243,
            243,   2438,  30816,    191, 226158,    229,  10807, 248075,      2],
        [     2, 256147,  11262,    467, 140990, 248079,   3413,    393,    217,
           3070,  11365, 248079, 148828,   3430, 209476,  15452,    673,  53391,
         248079,   6459, 146168,  65888,  42369,   1159,   9318, 248079,  26733,
          11344, 233199, 24

In [13]:
k = 2
while True:
    latency_input_ids = inputs['input_ids'][:, :k]
    latency_attention_mask = inputs["attention_mask"][:, :k]
    start_target = torch.tensor([[model.config.decoder_start_token_id, START_TOKEN]], dtype=torch.long, device=model.device)

    outputs = torch.argmax(
        model(
            input_ids=torch.concat([latency_input_ids, torch.tensor([[tokenizer.eos_token_id]], device=model.device)], dim=-1),
            attention_mask=torch.concat([latency_attention_mask, torch.tensor([[1]], dtype=torch.long, device=model.device)], dim=-1),
            decoder_input_ids=start_target,
            use_cache=False
        ).logits[:, -1], dim=-1
    ).item()
    print(f"k = {k}")
    print(f"\tPrefix: {tokenizer.batch_decode(latency_input_ids, skip_special_tokens=True)[0]}")
    print(f"\tGenerated token: {tokenizer.convert_ids_to_tokens(outputs)}")

    k += 1

    if k >= inputs['input_ids'].shape[1]:
        break

k = 2
	Prefix: When
	Generated token: ▁Когда
k = 3
	Prefix: When the
	Generated token: ▁Когда
k = 4
	Prefix: When the old
	Generated token: ▁Когда
k = 5
	Prefix: When the old man
	Generated token: ▁Когда
k = 6
	Prefix: When the old man opened
	Generated token: ▁Когда
k = 7
	Prefix: When the old man opened the
	Generated token: ▁Когда
k = 8
	Prefix: When the old man opened the small
	Generated token: ▁Когда
k = 9
	Prefix: When the old man opened the small wo
	Generated token: ▁Когда
k = 10
	Prefix: When the old man opened the small wooden
	Generated token: ▁Когда
k = 11
	Prefix: When the old man opened the small wooden door
	Generated token: ▁Когда
k = 12
	Prefix: When the old man opened the small wooden door at
	Generated token: ▁Когда
k = 13
	Prefix: When the old man opened the small wooden door at the
	Generated token: ▁Когда
k = 14
	Prefix: When the old man opened the small wooden door at the end
	Generated token: ▁Когда
k = 15
	Prefix: When the old man opened the small wooden door 

In [ ]:
target_tokens

In [2]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_transformer_distillation")

<Experiment: artifact_location='/mlflow/artifacts/1', creation_time=1779186531684, experiment_id='1', last_update_time=1779186531684, lifecycle_stage='active', name='simulmt_waitk_transformer_distillation', tags={}, trace_location=None, workspace='default'>

In [9]:
results = mlflow.search_runs(experiment_ids=["simulmt_waitk_transformer_distillation"])

MlflowException: API request to http://localhost:5000/api/2.0/mlflow/runs/search failed with exception HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /api/2.0/mlflow/runs/search (Caused by ResponseError('too many 500 error responses'))

In [3]:
from pathlib import Path

print(Path.cwd())
print(Path("./models/mamba2attn.bin").resolve())
print(Path("./models/mamba2attn.bin").exists())
print(Path("./mamba2attn.bin").resolve())
print(Path("./mamba2attn.bin").exists())

/root/LinearSimultMT
/root/blobs/730142f91e21e6225291625c3f7cbe8fbd2ccc824bf567b13aadead4d2909354
False
/root/LinearSimultMT/mamba2attn.bin
False


In [6]:
import torch
model = torch.load("/root/.cache/huggingface/hub/models--state-spaces--mamba2attn-2.7b/snapshots/5e0f47f0003095d6bdda3ad6fd7f3f41f274accb/pytorch_model.bin")

In [7]:
model

OrderedDict([('backbone.embedding.weight',
              tensor([[-0.2184,  0.0422,  0.3079,  ..., -0.0284, -0.0568,  0.0891],
                      [-0.0451, -0.0225,  0.0110,  ...,  0.0513, -0.0134, -0.0070],
                      [-0.0471, -0.0025,  0.1754,  ..., -0.1144,  0.0052,  0.0936],
                      ...,
                      [-0.0387,  0.0110, -0.0078,  ...,  0.0197, -0.0140, -0.0056],
                      [-0.0262, -0.0329, -0.0007,  ...,  0.0362, -0.0320,  0.0247],
                      [-0.0177, -0.0138,  0.0239,  ...,  0.0105, -0.0487, -0.0130]],
                     device='cuda:0', dtype=torch.float16)),
             ('backbone.layers.0.norm.weight',
              tensor([0.2238, 0.2168, 0.2322,  ..., 0.2157, 0.2106, 0.2191], device='cuda:0',
                     dtype=torch.float16)),
             ('backbone.layers.0.mixer.dt_bias',
              tensor([-5.1367, -5.3555, -4.5664, -5.3125, -4.2031, -5.5078, -4.9492, -5.2227,
                      -4.9609, -5.35